## 📑 Core Collections Report
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQDoHg)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col, regexp_replace
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import boto3
import os

# spark.conf.set("spark.sql.ansi.enabled", "false")

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(f'''
                    SELECT last_day(add_months({run_date!r}, -1)) AS SOM
                         ''').collect()[0]['SOM']
som_date = som_date.strftime('%Y-%m-%d')

som_date_wd = spark.sql(
    f'''
    Select MAX(CalendarDate)  SOMDate_WD
    from `{env_prefix}-cdh`.reference.ref_calendar
    WHERE LastDateOfMonth = last_day(add_months({run_date!r},-1)) and IsBusinessDay = 1
    '''
).collect()[0]['SOMDate_WD']

###### Schema of the table

In [0]:
table_name = 'Core_CollectionsReport' 

schema = (
"BusinessDate date," 
"ExposureKey string,"
"EBAStatus string,"
"CorePortfolio string,"
"AssignmentChannel string,"
"AssignmentChannelDetailed string,"
"AssignmentSegmentDetailed string,"
"COA string,"
"Anchor string,"
"SavedDay string,"
"DoubleTreatmentDNCHubs string,"
"OutflowSaved string,"
"RemainingDebt decimal(19,2),"
"FinalRiskSaved string,"
"FinalSaved string,"
"SavedReason string,"
"AssetSegment string,"
"CDI string,"
"CurrentBucket int,"
"ObjectiveStatus string,"
"ObjectiveStatusDate date,"
"SoMDebt decimal(19,2),"
"SoMBalance decimal(19,2),"
"SomBucket int,"
"InflowType string,"
"Installment decimal(19,2),"
"Outflow string,"
"RiskBucket string,"
"Cat string,"
"TargetType2 string,"
"Payments decimal(19,2),"
"SavedBalance decimal(19,2),"
"EBACat string,"
"InternalTeams string,"
"InternalTeamCode string,"
"TargetOutflowPercentage decimal(19,2),"
"TargetOutflowBalance decimal(19,2),"
"DTTargetCollection decimal(19,2),"
"NewTargetCollectionBalance decimal(19,2),"
"ExternalTargetRate decimal(19,2),"
"ExternalTarget decimal(19,2),"
"NewSegment string,"
"FinalSegment string,"
"Assign string,"
"AssignDetail string,"
"TargetPercentage decimal(19,2),"
"TargetBalance decimal(19,2),"
"ForTable2 string,"
"Description string,"
"Description2 string,"
"AttemptedBalance decimal(19,2),"
"NotFoundBalance decimal(19,2),"
"ContactedBalance decimal(19,2),"
"RPCBalance decimal(19,2),"
"PromiseBalance decimal(19,2),"
"PositivetoModifyBalance decimal(19,2),"
"NegativeBalance decimal(19,2),"
"RestRPCCommunicationsBalance decimal(19,2),"
"KeptBalance decimal(19,2),"
"BrokenBalance decimal(19,2),"
"PendingBalance decimal(19,2),"
"CancelledBalance decimal(19,2),"
"ExternalTargetRateCalc decimal(19,2),"
"ActivityDate timestamp,"
"FinalAssignmentChannel string,"
"FinalAssignmentChannelDetailed string,"
"PaymentAtttributionChannel1 string,"
"PaymentAtttributionChannelDetailed1 string,"
"PaymentAtttributionAmount1 decimal(19,2),"
"PaymentAtttributionChannel2 string,"
"PaymentAtttributionChannelDetailed2 string,"
"PaymentAtttributionAmount2 decimal(19,2),"
"PaymentAtttributionChannel3 string,"
"PaymentAtttributionChannelDetailed_3 string,"
"PaymentAtttributionAmount3 decimal(19,2),"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### AssignmentMapping

In [0]:
AssignmentMapping_v0 = spark.sql(
f'''
    SELECT DISTINCT	
                    a.BusinessDate
                ,	a.ExposureKey
                ,   a.CDI				
                ,	a.EBAStatus
                ,	a.Anchor				
                ,	a.SavedDate				
                ,   CASE WHEN (lower(a.AssignmentSegmentDetailed) ILIKE '%extension%' OR AssignmentSegmentDetailed ILIKE '%LowInstallment%') 
                              AND {run_date!r} >= '2025-01-01' THEN 'Call Center' 
                         ELSE a.AssignmentChannel END AS AssignmentChannel
                ,   CASE WHEN (a.AssignmentSegmentDetailed ILIKE '%Extension%' OR a.AssignmentSegmentDetailed ILIKE '%LowInstallment%') AND  a.AssignmentChannel='IRT' THEN 'Call Center'
                         WHEN (a.AssignmentSegmentDetailed ILIKE '%pending assignment%' OR a.AssignmentSegmentDetailed ILIKE '%SMS Campaign Lazy Payers%') THEN NULL     
                    ELSE a.AssignmentTeam END AS AssignmentTeam
                ,	a.AssignmentSegmentDetailed
                ,   a.CourseOfAction
                ,	a.SavedQueues
                ,	a.AssetSegment
			 ,	a.SoMObjectiveStatus 
			 ,	a.SoMObjectiveDate 
			 ,	a.SoMBalanceAmountEUR 
			 ,	a.SoMSourceBucket 
                ,    a.RiskBucket
                ,	CASE WHEN a.Event IN ('Saved due to Restructure' , 'Saved due to Extension') AND AttributionChannel IS NOT NULL AND AttributionChannel <> 'NULL' THEN 
                         CASE WHEN a.AttributionChannel = 'Hub RMs' THEN 'HUB RMs' 
                              ELSE a.AttributionChannel END 
                         WHEN (a.AssignmentSegmentDetailed ILIKE '%Extension%' OR a.AssignmentSegmentDetailed ILIKE '%LowInstallment%') AND  a.AssignmentChannel='IRT' THEN 'Call Center'
                         WHEN (a.AssignmentSegmentDetailed ILIKE '%pending assignment%' OR a.AssignmentSegmentDetailed ILIKE '%SMS Campaign Lazy Payers%') AND  a.AssignmentChannel='Call Center' THEN NULL     
                    ELSE a.AssignmentChannel
                    END AS FinalAssignmentChannel
                ,	CASE WHEN a.Event IN ('Saved due to Restructure' , 'Saved due to Extension') AND AttributionChannel IS NOT NULL AND AttributionChannel <> 'NULL' THEN a.AttributionTeam ELSE a.AssignmentTeam END FinalAssignmentTeam
                , RANK() OVER (PARTITION BY ExposureKey ORDER BY CASE WHEN Event = 'Saved due to Restructure' THEN 1 
                                                                      WHEN Event = 'Restructure' THEN 2 
                                                                      WHEN Event = 'Saved due to Extension' THEN 3 
                                                                      WHEN Event = 'Extension' THEN 4 
                                                                      WHEN Event = 'Saved due to Payment' THEN 5
                                                                      WHEN Event = 'Payment' THEN 6
                                                                      WHEN Event = 'Saved due to Other Reason' THEN 7
                                                                      WHEN Event = 'Active Application' THEN 8
                                                                      WHEN Event = 'w/o Event' THEN 9
                                                                      WHEN Event is null THEN 10 END ASC) AS Rank
    FROM `{env_prefix}-cdh-delivery`.repoint.Core_AssignmentDataMapping a
    WHERE 1=1
    AND a.BusinessDate = {run_date!r} 
    AND (      
			 (CASE WHEN a.RiskBucket IS NULL THEN 0 ELSE a.RiskBucket END <=3) 
             OR
			 (CASE WHEN a.RiskBucket IS NULL THEN 0 ELSE a.RiskBucket END >=4 AND a.CourseOfAction IN ('Collection', 'Double Treatment', 'collection')) 
			)
'''
)
AssignmentMapping_v0.createOrReplaceTempView("AssignmentMapping_v0")

In [0]:
AttributionChannel_Pay = spark.sql(
f'''
	SELECT * ,  RANK() OVER (PARTITION BY ExposureKey ORDER BY CASE WHEN AttributionChannel = 'NULL' THEN NULL ELSE AttributionChannel END DESC, CASE WHEN AttributionTeam = 'NULL' THEN NULL ELSE AttributionTeam END DESC) Rank
	FROM (
	    SELECT DISTINCT
					ExposureKey
				,	AttributionChannel
				,	AttributionTeam
				,	SUM(CASE WHEN /*PaymentAmount = 'NULL' OR*/ PaymentAmount IS NULL /*OR PaymentAmount = ''*/ THEN 0 ELSE 
						CASE WHEN CAST(REPLACE(PaymentAmount, ',', '0.') AS NUMERIC(19,2)) > 1 
							 THEN CAST(REPLACE(PaymentAmount, ',', '.') AS NUMERIC(19,2)) 
							 ELSE CAST(REPLACE(PaymentAmount, ',', '0.') AS NUMERIC(19,2)) END  END) AS PaymentAmount
		FROM `{env_prefix}-cdh-delivery`.repoint.Core_AssignmentDataMapping
		WHERE 1=1
		AND BusinessDate = {run_date!r}   	 
		AND Event IN ('Saved due to Payment','Payment') 
		GROUP BY	ExposureKey
				,	AttributionChannel
				,	AttributionTeam
				--,	AttributionAgent
    	) a
'''
)
AttributionChannel_Pay.createOrReplaceTempView("AttributionChannel_Pay")

In [0]:
AssignmentMapping = spark.sql(
f'''
		SELECT DISTINCT	
				P.BusinessDate
			,	P.ExposureKey
			, 	P.CDI
			,	P.EBAStatus
			,	P.Anchor					
			,	P.SavedDate AS SavedDay					
 			,	CASE 
					WHEN P.CourseOfAction = 'Double Treatment' AND  (P.AssignmentChannel = 'Hub RMs' OR P.AssignmentChannel='HUB RMs') AND P.SavedQueues LIKE '%net_int' THEN 'Call Center'
					ELSE P.AssignmentChannel
				END AS AssignmentChannel
			,	P.AssignmentTeam AS AssignmentChannelDetailed  
			,	P.AssignmentSegmentDetailed
			,	P.FinalAssignmentChannel FinalAssignmentChannel_start
			,	CASE WHEN (P.AssignmentSegmentDetailed ILIKE '%pending assignment%' OR P.AssignmentSegmentDetailed ILIKE '%SMS Campaign Lazy Payers%') THEN 'Unhandled'
                     WHEN P.AssignmentSegmentDetailed = 'Double Treatment' THEN 'Double Treatment'
                     WHEN P.CourseOfAction IN ('N.3869', 'unhandled') AND C.CalendarDay <=10 THEN 'unhandled / N3869 Saved' 
                     WHEN P.AssignmentChannel IS NULL AND P.CourseOfAction NOT IN ('N.3869', 'unhandled') THEN 'Unhandled'
                     ELSE P.CourseOfAction END AS COA
			,	P.CourseOfAction
   			,	CASE 
					WHEN P.CourseOfAction = 'Double Treatment' AND  P.FinalAssignmentChannel = 'Call Center' THEN 'DNC_Collection'
					WHEN P.CourseOfAction = 'Double Treatment' AND  lower(P.FinalAssignmentChannel) = lower('HUB RMs') THEN 'Hubs'
					ELSE NULL
				END AS `DOUBLE TREATMENT DNC-HUBS`
			,	CASE 
					WHEN P.CourseOfAction = 'Double Treatment' AND  P.FinalAssignmentChannel = 'HUB RMs' AND P.SavedQueues LIKE '%net_int' THEN 'Call Center'
					ELSE P.FinalAssignmentChannel
				END AS FinalAssignmentChannel
			,	P.FinalAssignmentTeam AS FinalAssignmentChannelDetailed
			,	P.AssetSegment
			,	P.SoMObjectiveStatus 
			,	P.SoMObjectiveDate 
			,	P.SoMBalanceAmountEUR 
			,	P.SoMSourceBucket 
   			, P.RiskBucket
      ,CASE WHEN 
							(CASE WHEN SoMSourceBucket > 3 THEN '4+' ELSE cast(SoMSourceBucket as STRING) END) IS NULL 
							OR (CASE WHEN SoMSourceBucket > 3 THEN '4+' ELSE cast(SoMSourceBucket as STRING) END) = 'NULL'
					THEN '' 
					ELSE (CASE WHEN SoMSourceBucket > 3 THEN '4+' ELSE cast(CAST(SoMSourceBucket AS INT)as STRING) END) END SOM_BUCKETForType
			,CASE WHEN 
							(CASE WHEN EBAStatus IN ('FNPL', 'NPL') THEN '(F)NPL' ELSE EBAStatus  END) IS NULL
							OR (CASE WHEN EBAStatus IN ('FNPL', 'NPL') THEN '(F)NPL' ELSE EBAStatus  END) = 'NULL'
					THEN ''
					ELSE (CASE WHEN EBAStatus IN ('FNPL', 'NPL') THEN '(F)NPL' ELSE EBAStatus  END) END AS EBAStatusForType
		FROM AssignmentMapping_v0 P
		LEFT JOIN `{env_prefix}-cdh`.reference.ref_calendar C ON C.CalendarDate = P.SavedDate
		WHERE Rank = 1
'''
)
AssignmentMapping.createOrReplaceTempView("AssignmentMapping")

%md
##### ReportUnified

In [0]:
extreport_unified_cl = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_cl WHERE BusinessDate = {run_date!r})
''')
extreport_unified_cl.createOrReplaceTempView("extreport_unified_cl")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_cl.count() == 0:
    print(f"The required external file extreport_unified_cl dataset for date {run_date!r} is empty")

extreport_unified_ml = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_ml WHERE BusinessDate = {run_date!r})
''')
extreport_unified_ml.createOrReplaceTempView("extreport_unified_ml")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_ml.count() == 0:
    print(f"The required external file extreport_unified_ml dataset for date {run_date!r} is empty")

extreport_unified_sb = spark.sql(
f'''		
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb
WHERE BusinessDate = {run_date!r}
AND IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp FROM `{env_prefix}-cdh`.externalfiles.extreport_unified_sb WHERE BusinessDate = {run_date!r})
''')
extreport_unified_sb.createOrReplaceTempView("extreport_unified_sb")     

# Check if the dataset is empty and print a message accordingly
if extreport_unified_sb.count() == 0:
    print(f"The required external file extreport_unified_sb dataset for date {run_date!r} is empty")



In [0]:
Report_Unified_v0 = spark.sql(f"""
SELECT 
	BusinessDate,
    ExposureKey,
    'CL' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    UnifiedApplicationStatus AS ApplicationStatus,
    TotalOutstandingAmount,
    FirstDateReceiptValuation AS FirstDateAtValidationFinal,
    ReceiptDateCurrentStep AS ApplicationCurrentStepInsertDateFinal,
    ApplicationImplementationDate  AS ApplicationImplementationDateFinal
FROM extreport_unified_cl
WHERE BusinessDate={run_date!r}
UNION ALL
 
SELECT 
    BusinessDate,
    ExposureKey,
    'ML' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    UnifiedApplicationStatus AS ApplicationStatus,
    TotalOutstandingAmount,
    FirstDateReceiptValuation as FirstDateAtValidationFinal,
    CurrentStepDate as ApplicationCurrentStepInsertDateFinal,
    ApplicationImplementationDate AS ApplicationImplementationDateFinal
FROM extreport_unified_ml
WHERE BusinessDate={run_date!r}

UNION ALL
 
SELECT BusinessDate,
    ExposureKey,
    'SB' as AssetSegment,
    ApplicationNumber as ApplicationNumber,
    CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate,
    UnifiedApplicationStatus AS ApplicationStatus,
    TotalOutstandingAmount,
    FirstDateReceiptValuation as FirstDateAtValidationFinal,
    CurrentStepReceiptDate as ApplicationCurrentStepInsertDateFinal,
    ApplicationImplementationDate AS ApplicationImplementationDateFinal 
FROM extreport_unified_sb
WHERE BusinessDate={run_date!r}
""")

Report_Unified_v0.createOrReplaceTempView("Report_Unified_v0")

In [0]:

#Report_Unified_v0 = spark.sql(
#f'''
#SELECT EU.*
#     /* , CASE WHEN EU.AssetSegment = 'CL' THEN EU.FirstDateReceiptValuation 
#            WHEN EU.AssetSegment = 'ML' THEN ML.date_eval_received
#            WHEN EU.AssetSegment = 'SB' THEN SB.`First_Receipt_Date_ApprovalCenter 1`
#       END AS FirstDateAtValidationFinal
#     , CASE WHEN EU.AssetSegment = 'CL' THEN EU.ReceiptDateCurrentStep 
#            WHEN EU.AssetSegment = 'ML' THEN ML.received_date
#            WHEN EU.AssetSegment = 'SB' THEN SB.InsertionDateToTheCurrentStep
#       END AS ApplicationCurrentStepInsertDateFinal
#     , CASE WHEN EU.AssetSegment = 'CL' THEN EU.ApplicationImplementationDate 
#            WHEN EU.AssetSegment = 'ML' THEN ML.`column 138`
#            WHEN EU.AssetSegment = 'SB' THEN SB.`column 151`
#  END AS ApplicationImplementationDateFinal */
#FROM ReportUnified_temp eu
#WHERE EU.BusinessDate={run_date!r}
#'''
#)
#Report_Unified_v0.createOrReplaceTempView("Report_Unified_v0")


In [0]:
Report_Unified = spark.sql(
f'''
	SELECT * FROM (
		SELECT * , RANK() OVER (PARTITION BY AccountNumber ORDER BY ApplicationCurrentStepInsertDate DESC, FirstDateAtValidation DESC) Rank
		FROM (
			SELECT AssetSegment AS BU
				 , ExposureKey AS AccountNumber
				 , ApplicationNumber
				 , CAST(CONCAT('20', left(ApplicationNumber,2), '-', SUBSTRING(ApplicationNumber,3,2), '-', SUBSTRING(ApplicationNumber,5,2)) AS DATE) AS ApplicationInsertDate
				 , ApplicationStatus
				 , FirstDateAtValidationFinal AS FirstDateAtValidation
				 , ApplicationCurrentStepInsertDateFinal AS ApplicationCurrentStepInsertDate
				 , ApplicationImplementationDateFinal
			FROM Report_Unified_v0
			WHERE --AssetSegment = 'CL' AND
				ApplicationNumber IS NOT NULL AND ApplicationNumber <> '' AND
				CASE WHEN TotalOutstandingAmount = '' THEN 0 ELSE CAST(TotalOutstandingAmount AS DECIMAL(19,2)) END>0
			AND ApplicationStatus='Ενεργή'
			AND (
				(AssetSegment = 'CL' 
				AND FirstDateAtValidationFinal IS NOT NULL AND FirstDateAtValidationFinal <> ' ' 
				AND (ApplicationImplementationDateFinal IS NULL OR ApplicationImplementationDateFinal=' ' OR ApplicationImplementationDateFinal=''))
				OR (AssetSegment = 'ML' 
				AND FirstDateAtValidationFinal IS NOT NULL AND FirstDateAtValidationFinal <> ' ' 
				AND (ApplicationImplementationDateFinal IS NULL OR ApplicationImplementationDateFinal=' ' OR ApplicationImplementationDateFinal=''))
				OR (AssetSegment = 'SB' 
				AND FirstDateAtValidationFinal IS NOT NULL AND FirstDateAtValidationFinal <> ' ' 
				AND (ApplicationImplementationDateFinal IS NULL OR ApplicationImplementationDateFinal=' ' OR ApplicationImplementationDateFinal=''))
				)
		) A
    ) B WHERE Rank = 1
'''
)
Report_Unified.createOrReplaceTempView("Report_Unified")

##### DailyPortfolio

In [0]:
DailyPortfolio = spark.sql(
f'''
	  SELECT 
				A.ACCOUNT_NUMBER
			,	A.CASE_BU					AS BU --Assumption: account will not change BU or CIF intramonth. Use SoM to not lose account that closed intramonth
			,	A.CUST_CIF				    AS CIF
			,	A.CASE_BUCKET				AS CURRENT_BUCKET
			,	A.CASE_DEBT_AMT			    AS CURRENT_DEBT
			,	A.CASE_OUTSTANDING_AMT	    AS CURRENT_BALANCE
			,	A.OBJECTIVE_STATUS		    AS OBJECTIVE_STATUS
			,	A.OBJECTIVE_STATUS_DATE	    AS OBJECTIVE_STATUS_DATE
      FROM `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio A 
      WHERE A.BusinessDate = {run_date!r} 
'''
)
DailyPortfolio.createOrReplaceTempView("DailyPortfolio")

In [0]:
SoMDailyPortfolio = spark.sql(
f'''
	  SELECT 
				A.ACCOUNT_NUMBER
			,	A.CASE_BU					AS BU --Assumption: account will not change BU or CIF intramonth. Use SoM to not lose account that closed intramonth
			,	A.CUST_CIF				    AS CIF
			,	A.CASE_BUCKET				AS SOM_BUCKET
			,	A.CASE_DEBT_AMT			    AS SOM_DEBT
			,	A.CASE_OUTSTANDING_AMT	    AS SOM_BALANCE
			,	A.OBJECTIVE_STATUS		    AS OBJECTIVE_STATUS
			,	A.OBJECTIVE_STATUS_DATE	    AS OBJECTIVE_STATUS_DATE
      FROM `{env_prefix}-cdh`.landed.repointqcab_daily_portfolio A
      WHERE A.BusinessDate = {som_date_wd!r} 
'''
)
SoMDailyPortfolio.createOrReplaceTempView("SoMDailyPortfolio")

##### ExposuresN

In [0]:
ExposuresN_Step0 = spark.sql(
f'''
	  SELECT
				E.Account_Number
			,	E.MODIFICATION_INSTALLMENT
      , E.REF_DATE
      , rank() over (partition by E.Account_Number ORDER BY CASE WHEN E.REF_DATE LIKE '%-%' THEN to_date(LEFT(E.REF_DATE, 10), 'yyyy-MM-dd') ELSE to_date(try_to_timestamp(replace(replace(regexp_replace(E.REF_DATE, '  ', ' '), ' AM', 'AM'), ' PM', 'PM'), 'MMM d yyyy hh:mma')) END desc, CAST(E.BusinessDate AS DATE) desc) as rank
      FROM `{env_prefix}-cdh`.landed.repointqcab_exposuresn E
      WHERE CASE WHEN E.REF_DATE LIKE '%-%' THEN to_date(LEFT(E.REF_DATE, 10), 'yyyy-MM-dd') ELSE to_date(try_to_timestamp(replace(replace(regexp_replace(E.REF_DATE, '  ', ' '), ' AM', 'AM'), ' PM', 'PM'), 'MMM d yyyy hh:mma')) END > {som_date_wd!r} AND CASE WHEN E.REF_DATE LIKE '%-%' THEN to_date(LEFT(E.REF_DATE, 10), 'yyyy-MM-dd') ELSE to_date(try_to_timestamp(replace(replace(regexp_replace(E.REF_DATE, '  ', ' '), ' AM', 'AM'), ' PM', 'PM'), 'MMM d yyyy hh:mma')) END <= {run_date!r}

'''
)
ExposuresN_Step0.createOrReplaceTempView("ExposuresN_Step0")

In [0]:
ExposuresN = spark.sql(
f'''
	  SELECT 
				Account_Number
			,	MODIFICATION_INSTALLMENT
      FROM ExposuresN_Step0
	  WHERE rank =1
'''
)
ExposuresN.createOrReplaceTempView("ExposuresN")

##### UnionFile

In [0]:
UnionFile = spark.sql(
f'''
	  SELECT 
				U.SnapshotDate
			,	U.ExposureNumber
      , U.ExposureKey
			,	U.NonCoreFlag
			,	CASE WHEN U.SnapshotDate = LAST_DAY(DATEADD(MONTH, -1, {run_date!r})) THEN CAST(U.BucketCategoryWithMateriality AS INT) ELSE NULL  END AS RiskBucket
      , U.EBAStatus
      , U.FPLUnderProbationFlag
      FROM `{env_prefix}-cdh`.externalfiles.extunionfile U
      WHERE U.SnapshotDate = {som_date!r}
      AND U.IngestionTimeStamp = (SELECT MAX(IngestionTimeStamp) 
                        FROM `{env_prefix}-cdh`.externalfiles.extunionfile 
                        WHERE SnapshotDate = {som_date!r})
'''
)
UnionFile.createOrReplaceTempView("UnionFile")

# Check if the dataset is empty and print a message accordingly
if UnionFile.count() == 0:
    print(f"The required external file extunionfile dataset for date {som_date!r} is empty")

##### PaymentsMtD

In [0]:
PaymentsMtD = spark.sql(
f'''

	   SELECT 
				PM.ExposureKey
			,	PM.PaymentAmount	
	   FROM `{env_prefix}-cdh-delivery`.repoint.core_paymentsmtd PM  
	   WHERE PM.BusinessDate = {run_date!r}
'''
)
PaymentsMtD.createOrReplaceTempView("PaymentsMtD")

##### SavedFile

In [0]:
SavedFile = spark.sql(
f'''
	   SELECT 
					S.Account
				,	S.OutflowSaved
				--,	CASE WHEN S.MLRemainingDebt IS NULL OR S.MLRemainingDebt = 'NULL' THEN 0 ELSE CAST(REPLACE(S.MLRemainingDebt, ',', '.') AS DECIMAL(19,2)) END   AS MLRemainingDebt 
				--,	CASE WHEN S.LMSRemainingDebt IS NULL OR S.LMSRemainingDebt = 'NULL' THEN 0 ELSE CAST(REPLACE(S.LMSRemainingDebt, ',', '.') AS DECIMAL(19,2)) END AS LMSRemainingDebt 
                --,   CASE WHEN S.MLRemainingDebt IS NULL OR S.MLRemainingDebt = 'NULL' THEN 0 ELSE CAST(REPLACE(S.MLRemainingDebt, ',', '.') AS DECIMAL(19,2)) END + CASE WHEN S.LMSRemainingDebt IS NULL OR S.LMSRemainingDebt = 'NULL' THEN 0 ELSE CAST(REPLACE(S.LMSRemainingDebt, ',', '.') AS DECIMAL(19,2)) END AS RemainingDebt
                ,   NULL AS MLRemainingDebt
                ,   NULL AS LMSRemainingDebt
                ,   NULL AS RemainingDebt
				,	S.FinalRiskSaved  
				,	S.SavedFlag AS FinalSaved  
				,	S.SavedReason
				,	S.InflowType
	   FROM `{env_prefix}-cdh`.derivative.dportfoliotoolcore S 
	--    FROM `{env_prefix}-cdh-delivery`.repoint.dportfoliotoolcore_uat_27 S 
       WHERE S.BusinessDate = {run_date!r} AND S.RelationType = 'Primary Owner'
'''
)
SavedFile.createOrReplaceTempView("SavedFile")

##### dod_dpd_fcy

In [0]:
dod_dpd_fcy = spark.sql(f"""
WITH extdod_dpd_fcy AS (
    SELECT
        AC_CODE as ExposureKey,
        AC_SRC_STM_CODE as ExposureSourceSystem,
        BusinessDate,
        COALESCE(
            TRY_TO_DATE(
                REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                'ddMMMyyyy'
            ),
            TRY_TO_DATE(
                REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                'dMMMyyyy'
            ),
            TRY_TO_DATE(
                TRIM(DOD_DPD_CNTER_STRT_DT),
                'd/M/yyyy'
            )
        ) AS DODDPDStartDate,

        COALESCE(
            TRY_TO_DATE(
                REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                'ddMMMyyyy'
            ),
            TRY_TO_DATE(
                REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                'dMMMyyyy'
            ),
            TRY_TO_DATE(
                TRIM(DOD_DPD_CNTER_END_DT),
                'd/M/yyyy'
            )
        ) AS DODDPDEndDate
    FROM `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy 
)

SELECT ExposureKey,
       DODDPDStartDate,
       DODDPDEndDate,
       ExposureSourceSystem,
       BusinessDate
FROM extdod_dpd_fcy
WHERE BusinessDate = {run_date!r}
AND MONTH(DODDPDStartDate) =  MONTH({run_date!r}) 
AND YEAR(DODDPDStartDate) = YEAR({run_date!r}) 
""")
dod_dpd_fcy.createOrReplaceTempView("dod_dpd_fcy")

##### collection_offer

In [0]:
collection_offer = spark.sql(
f'''
select * from (
    select * from (
        select  co.Exposurekey, co.BusinessDate as MaxBusinessDate, `_update_timestamp` as MaxUpdateTimeStamp, co.Step_2_segment_mapped_values, RANK() OVER (PARTITION BY co.ExposureKey ORDER BY `_update_timestamp` DESC) AS Rank
        from `{env_prefix}-ptools`.assignments.offer_collection co
        left join (
            select e.Exposurekey, case when e.EBAStatus = 'FPL' and e.FPLUnderProbationFlag = 1 then 'CFPL' else e.EBAStatus end EBAStatus, e.SnapshotDate
          from unionfile e 
        ) uf 
        on uf.ExposureKey = co.ExposureKey
        where 1=1 
        --and Step_2_segment_mapped_values LIKE '%CFPLProActiveInflow%'
        and MONTH(co.BusinessDate) =  MONTH({run_date!r}) 
        and YEAR(co.BusinessDate) = YEAR({run_date!r}) 
        and MONTH(`_update_timestamp`) =  MONTH({run_date!r}) 
        and YEAR(`_update_timestamp`) = YEAR({run_date!r}) 
        and coalesce(uf.EBAStatus, co.EBAStatus) = 'CFPL'
    )
    WHERE rank = 1
)
WHERE Step_2_segment_mapped_values LIKE '%CFPLProActiveInflow%'
'''
)
collection_offer.createOrReplaceTempView("collection_offer")

##### InflowType

In [0]:
InflowType = spark.sql(
f'''
SELECT DISTINCT 
	P.ExposureKey
,	I.DOD_Bucket AS RiskBucketMTD
,	I.MaxInflowFlag
,	I.InflowMonth
,	CASE 
	WHEN (P.EBAStatus = 'CFPL' and P.RiskBucket = 0 and dod.Exposurekey IS NOT NULL) OR (P.EBAStatus = 'CFPL' and co.MaxBusinessDate IS NOT NULL and co.MaxUpdateTimestamp IS NOT NULL) THEN 'Proactive NPE Inflow'
	WHEN S.InflowType IS NOT NULL AND S.InflowType <> 'NULL' THEN S.InflowType
	WHEN date_format({run_date!r}, 'yyyy-MM')=I.InflowMonth AND I.MaxInflowFlag IN ('PL Max Inflow', 'FPL Max Inflow', 'CFPL Max Inflow') THEN 'NPE Inflow'
	WHEN I.MaxInflowFlag IN ('PL Max Inflow', 'FPL Max Inflow', 'CFPL Max Inflow') THEN 'Proactive NPE Inflow'
	WHEN date_format({run_date!r}, 'yyyy-MM')=I.InflowMonth AND I.MaxInflowFlag IN ('NPL Max Inflow', 'FNPL Max Inflow') THEN 'NPL Inflow'
	WHEN S.InflowType IS NOT NULL AND S.InflowType <> 'NULL'
				OR (I.InflowMonth = date_format(last_day({run_date!r}), 'yyyy-MM') OR I.InflowMonth = date_format(last_day(date_add(last_day({run_date!r}), 1)), 'yyyy-MM')) THEN 'Proactive NPL Inflow'
	ELSE NULL
	END AS InflowType,
 CASE WHEN (P.EBAStatus = 'CFPL' and P.RiskBucket = 0 and dod.Exposurekey IS NOT NULL) OR (P.EBAStatus = 'CFPL' and co.MaxBusinessDate IS NOT NULL and co.MaxUpdateTimestamp IS NOT NULL) THEN 1 ELSE 0 END AS InflowTypeId
FROM AssignmentMapping P
LEFT JOIN dod_dpd_fcy dod on dod.Exposurekey = p.Exposurekey 
LEFT JOIN collection_offer co ON P.ExposureKey = co.ExposureKey 
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_maxinflows I 
		ON P.ExposureKey = I.ExposureNumber AND I.BusinessDate = {som_date_wd!r}
LEFT JOIN SavedFile S ON S.Account = P.ExposureKey
WHERE S.InflowType IS NOT NULL AND S.InflowType <> 'NULL'
OR (I.InflowMonth = date_format(last_day({run_date!r}), 'yyyy-MM') OR I.InflowMonth = date_format(last_day(date_add(last_day({run_date!r}), 1)), 'yyyy-MM'))
OR (dod.ExposureKey IS NOT NULL OR co.ExposureKey IS NOT NULL)
'''
)
InflowType.createOrReplaceTempView("InflowType")

##### Gefyra

In [0]:
Gefyra = spark.sql(
f'''
        SELECT DISTINCT Account_Number FROM (
            SELECT * FROM `{env_prefix}-cdh`.reference.extfile_implementations_gefyra_i_31_01_2023
            UNION ALL 
            SELECT * FROM `{env_prefix}-cdh`.reference.extfile_implementations_gefyra_ii_31_01_2023
        )
 '''
)
Gefyra.createOrReplaceTempView("Gefyra")

##### ExitPool

In [0]:
ExitPool = spark.sql(
f'''
	    SELECT  
				C.ExposureNumber
			,	C.ExitMonthGroupedNumber
			,	CASE 
					WHEN C.ExitMonthGroupedNumber = 'Expired' OR C.ExitMonthGroupedNumber = date_format({run_date!r}, 'yyyyMM') THEN 'Current Month'
					WHEN C.ExitMonthGroupedNumber =  date_format(last_day(add_months({run_date!r}, 1)), 'yyyyMM') THEN C.ExitMonthGroupedNumber
					WHEN C.ExitMonthGroupedNumber =  date_format(last_day(add_months({run_date!r}, 2)), 'yyyyMM') THEN C.ExitMonthGroupedNumber
					ELSE 'not'
				END AS ExitPool
	    FROM `{env_prefix}-cdh-delivery`.repoint.core_curings C 
     	WHERE C.BusinessDate = {run_date!r} 
 '''
)
ExitPool.createOrReplaceTempView("ExitPool")

##### extpaymentjumptotalperimeter

In [0]:
extpaymentjumptotalperimeter = spark.sql(
f'''
SELECT *
FROM `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter PJ 
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter GROUP BY BusinessDate) PJT
ON PJ.BusinessDate = PJT.BusinessDate AND PJ.IngestionTimeStamp = PJT.IngestionTimeStamp 
WHERE PJ.BusinessDate = (SELECT MAX(BusinessDate) FROM `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter WHERE BusinessDate <= {run_date!r})
 ''')
extpaymentjumptotalperimeter.createOrReplaceTempView("extpaymentjumptotalperimeter")        

# Check if the dataset is empty and print a message accordingly
if extpaymentjumptotalperimeter.count() == 0:
    print(f"The required external file extpaymentjumptotalperimeter dataset for dates <= {run_date!r} is empty")

##### Datatset

In [0]:
DataSet_step1 = spark.sql(
f'''
	   SELECT DISTINCT
				PR.ExposureKey
			,	PR.EBAStatus
			,	UN.NonCoreFlag AS CorePortfolio
			,	PR.AssignmentChannel
			,	PR.AssignmentChannelDetailed
			,	PR.AssignmentSegmentDetailed
			,	PR.COA
			,	PR.`DOUBLE TREATMENT DNC-HUBS`
			,	PR.Anchor
			,	PR.SavedDay
			,	SV.OutflowSaved
			,	SV.RemainingDebt
			,	SV.FinalRiskSaved
			,	SV.FinalSaved
			,	SV.SavedReason
			,	PR.AssetSegment AS BU
			,	PR.CDI AS CIF
			,	DP.CURRENT_BUCKET
			,	PR.SoMObjectiveStatus AS OBJECTIVE_STATUS
			,	PR.SoMObjectiveDate AS OBJECTIVE_STATUS_DATE
			,	SDP.SOM_DEBT
			,	PR.SoMBalanceAmountEUR AS SOM_BALANCE
			,	PR.SoMSourceBucket AS SOM_BUCKET
			
			, 	IT.InflowType 
			, 	IT.InflowTypeId	
			,	EX.MODIFICATION_INSTALLMENT AS Installment
			,	CASE WHEN EP.ExitPool IS NULL THEN 'not' ELSE EP.ExitPool END AS Outflow
			,	CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END AS RiskBucket
			,	CASE
					WHEN IT.InflowType = 'NPE Inflow' AND PR.EBAStatus = 'CFPL' THEN 'Risk 1'
					WHEN IT.InflowType IN ('NPE Inflow', 'NPL Inflow') THEN 'Risk 3'
					WHEN IT.InflowType IN ('Proactive NPE Inflow', 'Proactive NPL Inflow') AND PR.EBAStatus != 'CFPL' THEN 'Risk 2'
					WHEN IT.InflowType IN ('Proactive NPE Inflow') AND PR.EBAStatus = 'CFPL' THEN 'Proactive CFPL Inflow'
					WHEN CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END > 3 THEN 'Risk 4+'
					WHEN CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END > 0 THEN CONCAT('Risk ', CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END)
					WHEN PR.SoMSourceBucket > 3 THEN 'Op 4+'
					-- WHEN CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE PR.SoMSourceBucket END  > 3 THEN 'Op 4+'
					ELSE CONCAT('Op ', CASE WHEN PR.SoMSourceBucket IS NULL /*OR PR.SoMSourceBucket = 'NULL'*/ THEN '' ELSE CAST(CAST(PR.SoMSourceBucket AS INT)AS STRING) END)
				END	AS cat
			,	CASE 
					WHEN GF.Account_Number IS NOT NULL THEN 'Gefyra'
					WHEN PJ.ExposureKey IS NOT NULL AND PJ.PaymentJumpMonth <=  DATE_FORMAT(last_day(add_months({run_date!r}, 1)), 'yyyyMM') THEN 'PJ'
					ELSE 'BAU'
				END AS Target_Type
			,	IT.RiskBucketMTD
			,	CONCAT(
					CASE WHEN 
							(CASE WHEN CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END > 3 THEN '4+' ELSE cast(CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END as STRING) END) IS null 
							OR (CASE WHEN CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END > 3 THEN '4+' ELSE cast(CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END as STRING) END) = 'NULL'
					THEN '' 
					ELSE (CASE WHEN CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END > 3 THEN '4+' ELSE cast(CASE WHEN UN.RiskBucket IS NULL THEN IT.RiskBucketMTD ELSE UN.RiskBucket END as STRING) END) END
					, SOM_BUCKETForType
					, EBAStatusForType
					) AS Target_Type_2 
			,	CASE 
					WHEN CAST(REPLACE(PaymentAmount, ',', '.') AS DECIMAL(19,2)) > CAST(SDP.SOM_Debt AS DECIMAL(19,2)) THEN CAST(SDP.SOM_Debt AS DECIMAL(19,2))
					ELSE CAST(REPLACE(PaymentAmount, ',', '.') AS DECIMAL(19,2))
				END AS Payments
			, PM.PaymentAmount
			,	CASE 
					WHEN PR.EBAStatus = 'CFPL' AND SV.FinalSaved = 'Saved' AND SV.SavedReason IN ('Συμβατική Ρύθμιση', 'Ρύθμιση Haircut', 'Extension') THEN 0
					WHEN PR.`DOUBLE TREATMENT DNC-HUBS` = 'Hubs' AND PR.COA = 'Double Treatment' AND PR.FinalAssignmentChannel = 'Call Center'  AND SV.SavedReason IN ('Συμβατική Ρύθμιση', 'Ρύθμιση Haircut', 'Extension') THEN 0
					WHEN SV.FinalSaved = 'Saved' THEN CAST(CASE WHEN PR.SoMBalanceAmountEUR IS NULL THEN NULL ELSE PR.SoMBalanceAmountEUR END AS DECIMAL(19,2))
					ELSE 0
				END AS SavedBalance
			, 	EP.ExitPool
			,	CASE 
					WHEN PR.FinalAssignmentChannel = 'Call Center' AND IT.InflowType IS NOT NULL THEN 'InHouse Inflow'
					WHEN PR.FinalAssignmentChannel = 'Call Center' AND EP.ExitPool != 'not' THEN 'InHouse Outflow'
					WHEN PR.FinalAssignmentChannel = 'Call Center' THEN 'InHouse Rest'
					WHEN PR.FinalAssignmentChannel = 'Outsourcing Collection' AND /*CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE PR.SoMSourceBucket END*/ PR.SoMSourceBucket  = 1 THEN 'OutSource OpB1'
					WHEN PR.FinalAssignmentChannel = 'Outsourcing Collection' AND /*CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE PR.SoMSourceBucket END*/ PR.SoMSourceBucket = 2 THEN 'OutSource OpB2'
			 		WHEN PR.FinalAssignmentChannel = 'Outsourcing Collection' THEN 'OutSource OpB3+'
			 		ELSE '0'
			 	END AS `GIA PINAKA 2`
			/*,	CASE
					WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END  = 1 THEN 0.6
					WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END  = 2 THEN 0.3
			 		WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END  = 3 THEN 0.1
					ELSE 0
				END AS `Target Outflow (%)`*/
    		, 	CASE
					WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND /*CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END*/ PR.SoMSourceBucket  = 1 THEN 0.6
					WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND /*CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END*/ PR.SoMSourceBucket  = 2 THEN 0.3
			 		WHEN (EP.ExitPool != 'not'OR EP.ExitPool IS NULL) AND /*CASE WHEN PR.SoMSourceBucket = 'NULL' THEN NULL ELSE CAST(PR.SoMSourceBucket AS INT) END*/ PR.SoMSourceBucket  = 3 THEN 0.1
					ELSE 0
				END AS `Target Outflow (%)`
			,	PR.FinalAssignmentChannel
			,	FinalAssignmentChannelDetailed
	   FROM AssignmentMapping		PR
	   LEFT JOIN DailyPortfolio	DP ON PR.ExposureKey = DP.Account_Number
	   LEFT JOIN SoMDailyPortfolio	SDP ON PR.ExposureKey = SDP.Account_Number
	   LEFT JOIN ExposuresN		EX ON PR.ExposureKey = EX.Account_Number
	   LEFT JOIN UnionFile			UN ON PR.ExposureKey = UN.ExposureNumber
	   LEFT JOIN PaymentsMtD		PM ON PR.ExposureKey = PM.ExposureKey
	   LEFT JOIN InflowType		IT ON PR.ExposureKey = IT.ExposureKey
	   LEFT JOIN SavedFile			SV ON PR.ExposureKey = SV.Account
	   LEFT JOIN Gefyra			GF ON PR.ExposureKey = GF.Account_Number
	   LEFT JOIN ExitPool			EP ON PR.ExposureKey = EP.ExposureNumber
	   LEFT JOIN extpaymentjumptotalperimeter PJ ON PR.ExposureKey = PJ.ExposureKey
	  -- LEFT JOIN collection_offer co ON PR.ExposureKey = co.ExposureKey
      -- LEFT JOIN dod_dpd_fcy dod on dod.Exposurekey = PR.Exposurekey
    
    
 '''
)
DataSet_step1.createOrReplaceTempView("DataSet_step1")

In [0]:
CoreCollectionTargets = spark.sql(
f'''
	 SELECT * 
       FROM (
       SELECT *, rank() over(partition by BusinessDate order by IngestionTimestamp desc) as rank 
       FROM `{env_prefix}-reportinputfiles`.repoint.Core_CollectionTargets
       WHERE last_day(BusinessDate) = last_day({som_date!r})) A 
       WHERE A.rank = 1
 '''
)
CoreCollectionTargets.createOrReplaceTempView("CoreCollectionTargets")        

In [0]:
StaticCoreCollectionTeams = spark.sql(
f'''
	   SELECT *
       FROM `{env_prefix}-reportinputfiles`.repoint.core_staticcollectionteams
 '''
)
StaticCoreCollectionTeams.createOrReplaceTempView("StaticCoreCollectionTeams")        

In [0]:
DataSet_step2 = spark.sql(
f'''
 SELECT DISTINCT
				ST1.ExposureKey
			,	ST1.EBAStatus
			,	ST1.CorePortfolio
			--,	ST1.Core_Portfolio
			,	ST1.AssignmentChannel
			,	ST1.AssignmentChannelDetailed
			,	ST1.AssignmentSegmentDetailed
			,	ST1.COA
			,	ST1.Anchor
			,	ST1.SavedDay
--			,	ST1.AssignmentOutflow
			,	ST1.`DOUBLE TREATMENT DNC-HUBS`
			,	ST1.OutflowSaved
			,	ST1.RemainingDebt
			,	ST1.FinalRiskSaved
			,	ST1.FinalSaved
			,	ST1.SavedReason
			,	ST1.BU
			,	ST1.CIF
			,	ST1.CURRENT_BUCKET
			,	ST1.OBJECTIVE_STATUS
			,	ST1.OBJECTIVE_STATUS_DATE
			,	ST1.SOM_DEBT
			,	ST1.SOM_BALANCE
			,	ST1.SOM_BUCKET
			,	ST1.InflowType
			,   ST1.InflowTypeId
			,	ST1.Installment
			,	ST1.Outflow
			,	ST1.RiskBucket
			,	ST1.cat
			,	ST1.Target_Type
			,	ST1.Target_Type_2 
			,	ST1.Payments
			,	ST1.SavedBalance
			,	ST1.`Target Outflow (%)`
			,	ST1.`Target Outflow (%)` * CASE WHEN ST1.SOM_Balance IS NULL THEN NULL ELSE ST1.SOM_Balance END AS Target_Outflow_Balance
			,	CONCAT(CASE WHEN ST1.EBAStatus IS NULL OR ST1.EBAStatus = 'NULL' THEN '' ELSE ST1.EBAStatus END, ' ', ST1.cat) AS `EBA&CAT`
			,	CASE WHEN ST1.AssignmentChannel ='Call Center' AND ST1.AssignmentSegmentDetailed ='pipeline_PE' AND ST1.RiskBucket =0 THEN 'Rest_R0'
                     WHEN ST1.AssignmentChannel ='Call Center' AND ST1.AssignmentSegmentDetailed ='pipeline_PE' AND ST1.RiskBucket =1 THEN 'Rest_R1' 
					 WHEN ST1.InflowType = 'NPE Inflow' AND ST1.EBAStatus = 'CFPL' THEN 'CFPL_Inflow'
					 WHEN TM.TeamName = 'Outflow' AND ST1.EBAStatus <> 'FNPL' AND ST1.RiskBucket = 0 THEN 'Rest_R0'
					 WHEN TM.TeamName = 'Outflow' AND ST1.EBAStatus <> 'FNPL' AND ST1.RiskBucket = 1 THEN 'Rest_R1'
					 ELSE TM.TeamName 
					 END AS InternalTeams
			,	CASE 
					WHEN ST1.AssignmentChannel ='Call Center' AND ST1.AssignmentSegmentDetailed ='pipeline_PE' AND ST1.RiskBucket =0 THEN 'T3'
          WHEN ST1.AssignmentChannel ='Call Center' AND ST1.AssignmentSegmentDetailed ='pipeline_PE' AND ST1.RiskBucket =1 THEN 'T3'
          WHEN ST1.InflowType = 'NPE Inflow' AND ST1.EBAStatus = 'CFPL' THEN 'T1'
					WHEN TM.TeamName = 'Outflow' AND ST1.EBAStatus <> 'FNPL' AND ST1.RiskBucket = 0 THEN 'T3'
					WHEN TM.TeamName = 'Outflow' AND ST1.EBAStatus <> 'FNPL' AND ST1.RiskBucket = 1 THEN 'T3'
					ELSE TM.TeamCode 
					END AS InternalTeamCode
			,	CASE 
					WHEN ST1.COA = 'Double Treatment' AND ST1.`DOUBLE TREATMENT DNC-HUBS` = 'Hubs' THEN CT.DTCollection
					WHEN ST1.COA = 'Double Treatment'  THEN CT.DTTotal
					ELSE CT.CollectionTarget   
				END AS `DT Target Collection`
			,	ST1.`GIA PINAKA 2`
			,	ST1.FinalAssignmentChannel
			,	ST1.FinalAssignmentChannelDetailed
		FROM DataSet_step1 ST1
  		LEFT JOIN CoreCollectionTargets CT ON ST1.Target_Type_2 = CT.TargetType AND CT.BusinessDate = (SELECT MAX(BusinessDate) 
                                                                                                        FROM CoreCollectionTargets 
                                                                                                        WHERE BusinessDate <= {run_date!r})
		LEFT JOIN StaticCoreCollectionTeams TM ON COALESCE(ST1.`DOUBLE TREATMENT DNC-HUBS`, 'NULL') = COALESCE(TM.DoubleTreatmentDNCHUBS, 'NULL') AND ST1.AssignmentSegmentDetailed = TM.AssignmentSegmentDetailed
 '''
)
DataSet_step2.createOrReplaceTempView("DataSet_step2")

In [0]:
StaticCoreCollectionCategories = spark.sql(
f'''
select `EBA_CAT`,`Assign`, `AssignDetail`, `New_Segment`, `Final_Segment`
from `{env_prefix}-reportinputfiles`.repoint.core_staticcollectioncategories
-- from StaticCoreCollectionCategories
'''
)
StaticCoreCollectionCategories.createOrReplaceTempView("StaticCoreCollectionCategories")

In [0]:
DataSet_step3 = spark.sql(
f'''
SELECT DISTINCT 
                ST2.ExposureKey
			,	ST2.EBAStatus
			,	ST2.CorePortfolio
			,	ST2.AssignmentChannel
			,	ST2.AssignmentChannelDetailed
			,	ST2.AssignmentSegmentDetailed
			,	ST2.COA
			,	ST2.Anchor
			,	ST2.SavedDay
			,	ST2.`DOUBLE TREATMENT DNC-HUBS`
			,	ST2.OutflowSaved
			,	ST2.RemainingDebt
			,	ST2.FinalRiskSaved
			,	ST2.FinalSaved
			,	ST2.SavedReason
			,	ST2.BU
			,	ST2.CIF
			,	ST2.CURRENT_BUCKET
			,	ST2.OBJECTIVE_STATUS
			,	ST2.OBJECTIVE_STATUS_DATE
			,	ST2.SOM_DEBT
			,	ST2.SOM_BALANCE
			,	ST2.SOM_BUCKET
			,	ST2.InflowType
			,   ST2.InflowTypeId
			,	ST2.Installment
			,	ST2.Outflow
			,	ST2.RiskBucket
			,	ST2.cat
			,	ST2.Target_Type
			,	ST2.Target_Type_2 
			,	ST2.Payments
			,	ST2.SavedBalance
			,	ST2.`Target Outflow (%)`
			,	ST2.Target_Outflow_Balance			
			,	ST2.`EBA&CAT`
			,	ST2.InternalTeams
			,	ST2.InternalTeamCode
			-- ,	ST2.`DT Target Collection`                                                                          -- Missing from step2
			,	CASE WHEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) > 1 
   					 THEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '.') AS NUMERIC(19,6)) 
         			 ELSE CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) END `DT Target Collection` 
			,	CASE WHEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) > 1 
   					 THEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '.') AS NUMERIC(19,6)) 
         			 ELSE CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) END * CASE WHEN ST2.SOM_Balance IS NULL THEN NULL ELSE ST2.SOM_Balance END AS New_Target_Collection_Balance                         -- Missing from step2
			,	CASE				
   					WHEN ST2.InflowTypeId = 1 THEN 'Proactive CFPL Inflow'
					WHEN ST2.AssignmentSegmentDetailed = 'Oristikes_Apofaseis' AND ST2.AssignmentChannel ='Outsourcing Collection' THEN 'Oristikes_Apofaseis'
					WHEN ST2.AssignmentSegmentDetailed = 'OCW_Collection' AND ST2.AssignmentChannel ='Outsourcing Collection'      THEN 'OCW_Collection'
					WHEN ST2.cat = 'Proactive CFPL Inflow' THEN ST2.cat
					WHEN ST2.EBAStatus = 'CFPL' AND ST2.cat = 'Risk 1' THEN 'CFPE_Inflow'
					WHEN ST2.cat != 'Risk 4+' AND ST2.COA = 'Double Treatment' THEN 'Double Treatment'
					WHEN ST2.InflowType IN ('NPE Inflow', 'Proactive NPE Inflow') THEN 'NPE Inflow'
					WHEN ST2.InflowType IN ('NPL Inflow', 'Proactive NPL Inflow') THEN 'NPL Inflow'
					ELSE CC.`New_Segment`
				END AS New_Segment
			,	CASE WHEN ST2.InflowTypeId = 1 THEN 'Proactive CFPL Inflow'
					WHEN ST2.AssignmentSegmentDetailed = 'Oristikes_Apofaseis' AND ST2.AssignmentChannel ='Outsourcing Collection' THEN 'Oristikes_Apofaseis'
                    WHEN ST2.AssignmentSegmentDetailed = 'OCW_Collection' AND ST2.AssignmentChannel ='Outsourcing Collection'      THEN 'OCW_Collection'
                    WHEN ST2.cat = 'Proactive CFPL Inflow' THEN ST2.cat
					WHEN ST2.EBAStatus = 'CFPL' AND ST2.cat = 'Risk 1' THEN 'CFPE_Inflow'
					WHEN ST2.InflowType IN ('NPE Inflow', 'Proactive NPE Inflow') THEN 'NPE Inflow'
					WHEN ST2.InflowType IN ('NPL Inflow', 'Proactive NPL Inflow') THEN 'NPL Inflow'
					ELSE CC.`Final_Segment`
				END AS Final_Segment
			,	CASE WHEN ST2.InflowTypeId = 1 THEN 'Proactive NPE Inflow'
					WHEN ST2.InflowType IS NOT NULL AND ST2.InflowType <> 'NULL' THEN ST2.InflowType
					ELSE CC.Assign  
				END AS Assign
			,	CASE WHEN ST2.InflowTypeId = 1 THEN 'Proactive CFPL Inflow' 
					WHEN ST2.cat = 'Proactive CFPL Inflow' THEN ST2.cat
					WHEN ST2.InflowType IS NOT NULL AND ST2.InflowType <> 'NULL' THEN ST2.InflowType
					ELSE CC.AssignDetail  
				END AS AssignDetail
   			,	CASE WHEN ST2.AssignmentSegmentDetailed IN ('PJ_Extension', 'LowInstallment') AND ST2.FinalAssignmentChannel = 'Call Center' THEN 
						CASE WHEN CAST(REPLACE(CT.Extension, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Extension, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Extension, ',', '0.') AS NUMERIC(19,6)) END  
        			 WHEN ST2.AssignmentSegmentDetailed = 'CFPL_Inflow_Next_Month' AND ST2.FinalAssignmentChannel = 'Call Center' THEN 
						CASE WHEN CAST(REPLACE(CT.CFPLProactive, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.CFPLProactive, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.CFPLProactive, ',', '0.') AS NUMERIC(19,6)) END  
					 WHEN ST2.AssignmentSegmentDetailed = 'Outflow' AND EBAStatus = 'FNPL' AND ST2.Outflow != 'not' AND ST2.SOM_Bucket IN (1,2,3) AND ST2.RiskBucket < 2 THEN 
						CASE WHEN CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '0.') AS NUMERIC(19,6)) END
                     WHEN ST2.COA = 'Collection' AND ST2.FinalAssignmentChannel IN ('Call Center', 'Outsourcing Collection') THEN 
   						CASE WHEN CAST(REPLACE(CT.CollectionTarget, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.CollectionTarget, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.CollectionTarget, ',', '0.') AS NUMERIC(19,6)) END
					 WHEN ST2.COA IN ('Double Treatment', 'Restructuring') AND ST2.FinalAssignmentChannel IN ('Call Center', 'Outsourcing Collection') THEN 
   						CASE WHEN CAST(REPLACE(CT.DTTotal, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.DTTotal, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.DTTotal, ',', '0.') AS NUMERIC(19,6)) END
					 ELSE 
						CASE WHEN CAST(REPLACE(CT.Unhandled, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Unhandled, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Unhandled, ',', '0.') AS NUMERIC(19,6)) END
				END AS `Target %` --change 
            /*
			,	CASE 
					WHEN ST2.Outflow != 'not' AND ST2.AssignmentSegmentDetailed LIKE '%Outflow%' THEN 
						CASE WHEN CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(ST2.`Target Outflow (%)`, ',', '0.') AS NUMERIC(19,6)) END
					WHEN ST2.FinalAssignmentChannel = 'Call Center'  OR (ST2.FinalAssignmentChannel = 'Outsourcing Collection' AND (ST2.InflowType IS NULL OR ST2.InflowType = 'NULL')) THEN 
						CASE WHEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) END
					WHEN ST2.AssignmentSegmentDetailed = 'PJ_Extension' THEN 
						CASE WHEN CAST(REPLACE(CT.Extension, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Extension, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Extension, ',', '0.') AS NUMERIC(19,6)) END     
					WHEN ST2.COA = 'Restructuring' AND ST2.FinalAssignmentChannel = 'IRT' THEN 
     					CASE WHEN CAST(REPLACE(CT.Restructuring, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Restructuring, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Restructuring, ',', '0.') AS NUMERIC(19,6)) END
					WHEN ST2.COA = 'Restructuring' AND ST2.FinalAssignmentChannel = 'HUB RMs' THEN 
						CASE WHEN CAST(REPLACE(CT.Restructuring, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Restructuring, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Restructuring, ',', '0.') AS NUMERIC(19,6)) END
					ELSE CASE WHEN CAST(REPLACE(CT.Unhandled, ',', '0.') AS NUMERIC(19,6)) > 1 
							 THEN CAST(REPLACE(CT.Unhandled, ',', '.') AS NUMERIC(19,6)) 
							 ELSE CAST(REPLACE(CT.Unhandled, ',', '0.') AS NUMERIC(19,6)) END
				END AS `Target %`
            */
		    , CASE WHEN ST2.FinalAssignmentChannel = 'Outsourcing Collection' THEN 
      				CASE WHEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) > 1 
   					 	 THEN CAST(REPLACE(ST2.`DT Target Collection`, ',', '.') AS NUMERIC(19,6)) 
         			 	 ELSE CAST(REPLACE(ST2.`DT Target Collection`, ',', '0.') AS NUMERIC(19,6)) END 
              ELSE NULL END AS `Target % External`   -- Missing from step2
			, ST2.`GIA PINAKA 2`
			, ST2.FinalAssignmentChannel
			, ST2.FinalAssignmentChannelDetailed
		FROM DataSet_step2 ST2
		LEFT JOIN CoreCollectionTargets CT ON ST2.Target_Type_2 = CT.TargetType AND CT.BusinessDate = (SELECT MAX(BusinessDate) 
                                                                                                        FROM CoreCollectionTargets 
                                                                                                        WHERE BusinessDate <= {run_date!r})
		LEFT JOIN StaticCoreCollectionCategories CC ON ST2.`EBA&CAT` = CC.`EBA_CAT`
        --WHERE LOWER(AssignmentSegmentDetailed) NOT LIKE '%lazy%'
     '''
)
DataSet_step3.createOrReplaceTempView("DataSet_step3")

In [0]:
DataSet_step4 = spark.sql(
f'''
	   SELECT 	DISTINCT
	   			ST3.ExposureKey
			,	ST3.EBAStatus
			,	ST3.CorePortfolio
			,	ST3.AssignmentChannel
			,	ST3.AssignmentChannelDetailed
			,	ST3.AssignmentSegmentDetailed
			,	ST3.COA
			,	ST3.Anchor
			,	ST3.SavedDay
			,	ST3.`DOUBLE TREATMENT DNC-HUBS`
			,	ST3.OutflowSaved
			,	ST3.RemainingDebt
			,	ST3.FinalRiskSaved
			,	ST3.FinalSaved
			,	ST3.SavedReason
			,	ST3.BU
			,	ST3.CIF
			,	ST3.CURRENT_BUCKET
			,	ST3.OBJECTIVE_STATUS
			,	ST3.OBJECTIVE_STATUS_DATE
			,	ST3.SOM_DEBT
			,	ST3.SOM_BALANCE
			,	ST3.SOM_BUCKET
			,	ST3.InflowType
			,	ST3.Installment
			,	ST3.Outflow
			,	ST3.RiskBucket
			,	ST3.cat
			,	ST3.Target_Type
			,	ST3.Target_Type_2 
			,	ST3.Payments
			,	ST3.SavedBalance
			,	ST3.`EBA&CAT`
			,	ST3.InternalTeams
			,	ST3.InternalTeamCode
			,	ST3.`Target Outflow (%)`
			,	ST3.Target_Outflow_Balance
			,	ST3.`DT Target Collection`
			,	ST3.`DT Target Collection` * CASE WHEN ST3.SOM_Balance IS NULL THEN NULL ELSE ST3.SOM_Balance END AS New_Target_Collection_Balance
			-- ,	ST3.`Target % External`  AS External_Target_Rate --EX.External_Target_Rate  --VB 30.10.23
   			,	`Target % External`  AS External_Target_Rate --EX.External_Target_Rate  --VB 30.10.23
			,	`Target % External` * CASE WHEN ST3.SOM_Balance IS NULL THEN NULL ELSE ST3.SOM_Balance END AS External_Target   --VB 30.10.23
			,	ST3.New_Segment
			,	ST3.Final_Segment
			,	ST3.Assign
			,	ST3.AssignDetail
			,	ST3.`Target %`
			,	CASE WHEN ST3.SOM_Balance IS NULL THEN NULL ELSE ST3.SOM_Balance END * ST3.`Target %` AS `Target Balance`
			,	ST3.`GIA PINAKA 2`
			-- ,   CASE WHEN ST3.RiskBucket NOT IN ('0') THEN CONCAT(ST3.RiskBucket,ST3.EBAStatus)
			--          WHEN ST3.RiskBucket = 0 THEN CONCAT(CAST(ST3.RiskBucket AS STRING), 
            -- 												  CASE WHEN CASE WHEN ST3.SOM_BUCKET = 'NULL' THEN NULL ELSE ST3.SOM_BUCKET END IN (3,4,5,6,7,8,9,10,11,12,13) THEN '100' ELSE 
            -- 		 CASE WHEN ST3.SOM_BUCKET IS NULL OR ST3.SOM_BUCKET = 'NULL' THEN '' ELSE CAST(ST3.SOM_BUCKET AS STRING) END 
            --    		END, ST3.EBAStatus)
			--     END AS External_Target_Type_Calc 
			,   CASE WHEN ST3.RiskBucket <> 0 THEN CONCAT(CAST(ST3.RiskBucket AS STRING),ST3.EBAStatus)
			         WHEN ST3.RiskBucket = 0 THEN CONCAT(CAST(ST3.RiskBucket AS STRING), 
            												  CASE WHEN ST3.SOM_BUCKET IN (3,4,5,6,7,8,9,10,11,12,13) THEN '100' ELSE 
            		 CASE WHEN ST3.SOM_BUCKET IS NULL THEN '' ELSE CAST(ST3.SOM_BUCKET AS STRING) END 
               		END, ST3.EBAStatus)
			    END AS External_Target_Type_Calc 
			,	ST3.FinalAssignmentChannel
			,	ST3.FinalAssignmentChannelDetailed
		FROM DataSet_step3 ST3
     '''
)
DataSet_step4.createOrReplaceTempView("DataSet_step4")

##### ExternalTargetRateCalc

In [0]:
ExternalTargetRateCalc = spark.sql(
f'''
		SELECT 
				ST4.External_Target_Type_Calc 
			,	SUM(External_Target)/SUM(CASE WHEN ST4.SOM_BALANCE IS NULL THEN NULL ELSE ST4.SOM_BALANCE END) AS External_Target_Rate_Calc
		FROM DataSet_step4 ST4
		WHERE 1=1 
		AND ST4.FinalAssignmentChannel = 'Outsourcing Collection'
		GROUP BY ST4.External_Target_Type_Calc
     '''
)
ExternalTargetRateCalc.createOrReplaceTempView("ExternalTargetRateCalc")

##### interaction

In [0]:
interaction = spark.sql(
f'''
    select * from (
        select distinct ia.PortfolioName 
            ,   ia.ExposureKey
            ,   ia.InteractionTypeKey
            ,   ia.InteractionType
            ,   ia.InteractionDate
            ,   m.hierarchy
            ,   m.`3rd_level_category`
            ,   case when left(m.`2nd_level_category`,3) in('1.1','1.2') then left(m.`2nd_level_category`,3) else left(m.`1st_level_category`,1) end as Priority
            ,   m.Activity_Group
            ,   ia.PromiseStatus
            ,   ia.PromiseAmount
            ,   ia.PromiseToPayAmount
            ,   CAST(CONCAT(LEFT(ia.PromiseDueDate, 4), '-', RIGHT(LEFT(ia.PromiseDueDate, 6), 2), '-', RIGHT(ia.PromiseDueDate, 2)) AS DATE) AS PromiseDueDate
            ,   rank() over (partition by ia.ExposureKey, ia.InteractionTypeKey, ia.InteractionDate, ia.CreatedDate, ia.InteractionKey order by ia.BusinessDate asc) as rank -- Change FM 15/10/2025
            -- ,   rank() over (partition by ia.ExposureKey, ia.InteractionTypeKey, ia.InteractionDate, ia.CreatedDate, ia.InteractionKey order by ia.BusinessDate desc) as rank
            ,   ia.CreatedDate
            ,   ia.IsPromiseFlag
        from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) ia
        inner join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on ia.InteractionTypeKey = m.Activity_Code
        where 
        ia.InteractionDate > last_day({som_date_wd!r})
        and ia.InteractionDate <= {run_date!r}
        and m.Base_Category not in ('Legal Action', 'no attempt')
        and m.Activity_Group IN ('Modifications', 'Collections', 'Collections/Modifications') 
        and ia.InteractionTypeKey <> '100'
        and ia.PortfolioName = 'Core Retail'
        -- and ia.IsPromiseFlag = true
    ) a where rank = 1
'''
)
interaction.createOrReplaceTempView("interaction")

##### dialer

In [0]:
dialer = spark.sql(
f'''
select distinct
        de.ExposureKey
   ,   ' ' as InteractionTypeKey
   ,   cast(DateHour as date ) as InteractionDate
   ,   ' ' as `3rd_level_category`
   ,   1.3 Priority
   ,   DateHour AS CreatedDate
from `{env_prefix}-cdh`.primary.dialer d
join `{env_prefix}-cdh`.primary.exposure de on d.CaseCode = CAST(de.Casecode AS STRING) AND d.BusinessDate = de.BusinessDate AND de.sourcesystem = 'qca'
where de.PortfolioName = 'Core Retail'
and cast(d.DateHour as date ) > last_day({som_date_wd!r})
and cast(d.DateHour as date ) <= {run_date!r}
'''
)
dialer.createOrReplaceTempView("dialer")

##### interactionwithdialer

In [0]:
interactionwithdialer = spark.sql(
f'''
select *
    ,  rank() over (partition by ExposureKey, InteractionDate order by Hierarchy desc, CreatedDate DESC, `3rd_level_category` desc) AS rank  --, InteractionTypeKey asc) as rank
    ,   case when Priority = '1.1' then 'No Attempt'
             when Priority = '1.2' then 'Attempt'
             when Priority = '1.3' then 'Dialer Unsuccessful Attempt'
             when Priority = '2'   then 'Contact'
             when Priority = '3'   then 'Rest RPC Communication'
             when Priority = '4'   then 'Negative Communication'
             when Priority = '5'   then 'Positive Communication'
        end as StrongestAction
from (
select
        ExposureKey
    ,   InteractionTypeKey
    ,   InteractionDate
    ,   Priority
    ,   Hierarchy
    ,   `3rd_level_category`
    ,   Activity_Group
    ,   PromiseStatus
    ,   'Interaction' AS SourceFlag
    ,   CreatedDate
from interaction
-- where Rank = 1
union
select ExposureKey
    ,  InteractionTypeKey
    ,  InteractionDate
    ,  Priority
    ,  1.3 AS Hierarchy
    ,  `3rd_level_category`
    ,  NULL Activity_Group
    ,  NULL PromiseStatus
    , 'Dialer' AS SourceFlag
    ,  CreatedDate
from dialer
) interactionwithdialer
 
'''
)
interactionwithdialer.createOrReplaceTempView("interactionwithdialer")

In [0]:
strongestaction = spark.sql(
f'''
select *
from interactionwithdialer
where rank = 1
'''
)
strongestaction.createOrReplaceTempView("strongestaction")

##### Activities

In [0]:
Activities_01 = spark.sql(
f'''
		SELECT DISTINCT
				 LA.ExposureKey
				,LA.StrongestAction
				,AM.Base_Category AS BaseCategory
				,CASE WHEN LA.Hierarchy ='1.3' THEN 1.3 ELSE AM.Hierarchy END AS Hierarchy
				,CASE WHEN AM.1st_Level_Category = '0. Legal Action - no contact' THEN '0. Legal Action - no contact'
				      WHEN AM.1st_Level_Category = '1. no contact' AND AM.2nd_Level_Category = '1.1 no contact - no attempt'   THEN '1.1 no contact - no attempt'
				      WHEN AM.1st_Level_Category = '1. no contact' AND AM.2nd_Level_Category = '1.2 no contact - attempt'      THEN '1.2 no contact - attempt'
					  WHEN AM.1st_Level_Category = '2. contact - Not RPC'                                                      THEN '2. contact - Not RPC'
					  WHEN AM.1st_level_category = '3. Rest RPC Communications'                                                THEN '3. Rest RPC Communications'
					  WHEN AM.1st_level_category = '4. Negative Communication'                                                 THEN '4.Negative Communication'
					  WHEN AM.1st_level_category = '5.Positive Communication'                                                  THEN '4.Positive Communication'
                      WHEN LA.Hierarchy ='1.3'                                                                                 THEN '1.3 Dialer Unsuccessful Attempt'
				     END AS ReportingCategory
				,CASE WHEN AM.1st_Level_Category = '0. Legal Action - no contact'                                              THEN CAST('0'     AS STRING)
				      WHEN AM.1st_Level_Category = '1. no contact' AND AM.2nd_Level_Category = '1.1 no contact - no attempt'   THEN CAST('1.1'   AS STRING)
				      WHEN AM.1st_Level_Category = '1. no contact' AND AM.2nd_Level_Category = '1.2 no contact - attempt'      THEN CAST('1.2'   AS STRING)
					  WHEN AM.1st_Level_Category = '2. contact - Not RPC'                                                      THEN CAST('2'     AS STRING)
					  WHEN AM.1st_level_category = '3. Rest RPC Communications'                                                THEN CAST('3'     AS STRING)
					  WHEN AM.1st_level_category = '4. Negative Communication'                                                 THEN CAST('4'     AS STRING)
					  WHEN AM.1st_level_category = '5.Positive Communication'                                                  THEN CAST('4'     AS STRING)
                      WHEN LA.Hierarchy ='1.3'                                                                                 THEN CAST('1.3'   AS STRING)
				     END AS ReportingCategoryHierarchy
				,AM.1st_Level_Category AS 1stLevelCategory
				,AM.2nd_Level_Category AS 2ndLevelCategory
				,	CASE WHEN LA.PromiseStatus ='213' THEN 'Kept'
						 WHEN LA.PromiseStatus ='214' THEN 'Broken'
       					 WHEN LA.PromiseStatus ='215' THEN 'Pending'
					ELSE LA.PromiseStatus END AS PromiseStatus
 				,LA.InteractionDate
                ,LA.InteractionTypeKey 
		FROM strongestaction LA 
	    LEFT JOIN `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping AM ON AM.Activity_Code = LA.InteractionTypeKey
		WHERE 1=1
			  AND LA.InteractionDate > last_day(add_months({run_date!r}, -1)) 
			  AND LA.InteractionDate <=  {run_date!r} 
'''
)
Activities_01.createOrReplaceTempView("Activities_01")

In [0]:
Activities = spark.sql(
f'''
		SELECT DISTINCT
			  	  P.ExposureKey
				, P.StrongestAction
				, P.BaseCategory
				, P.Hierarchy
				, P.ReportingCategory
				, P.1stLevelCategory
				, P.2ndLevelCategory
				, P.PromiseStatus
				, RANK() OVER (PARTITION BY P.ExposureKey ORDER BY P.ReportingCategoryHierarchy DESC, P.InteractionDate DESC, P.InteractionTypeKey DESC ) AS Rank1
				, P.InteractionDate, P.InteractionTypeKey  
		FROM Activities_01 P
'''
)
Activities.createOrReplaceTempView("Activities")

In [0]:
PromiseStatus = spark.sql(
f'''
        SELECT DISTINCT  ExposureKey
       		           , PromiseStatus
        FROM (
       		SELECT DISTINCT 
       		       ExposureKey
       		     , PromiseStatus
       			 , InteractionDate
       			 , RANK() OVER (PARTITION BY P.ExposureKey ORDER BY P.InteractionDate DESC, CASE WHEN PromiseStatus = 'Pending' THEN 1
       			                                                                                   WHEN PromiseStatus = 'Broken' THEN 2
       																							   WHEN PromiseStatus = 'Kept' THEN 3 END DESC) AS Rank 
       		FROM Activities P
       		WHERE PromiseStatus IS NOT NULL
        ) A
        WHERE Rank=1
'''
)
PromiseStatus.createOrReplaceTempView("PromiseStatus")

##### StrongestActivity

In [0]:
StrongestActivity = spark.sql(
f'''
		SELECT DISTINCT
			 A.ExposureKey
			,A.StrongestAction
			,A.ReportingCategory
			,A.1stLevelCategory
			,A.2ndLevelCategory
			,P.PromiseStatus
		    ,A.InteractionDate  
		FROM Activities A
		LEFT JOIN PromiseStatus P ON P.ExposureKey = A.ExposureKey  
		WHERE A.Rank1 = 1
'''
)
StrongestActivity.createOrReplaceTempView("StrongestActivity")

##### FinalDataset

In [0]:
DataSet_step5 = spark.sql(
f'''
		 SELECT DISTINCT
 	   		 	ST4.ExposureKey
			,	ST4.EBAStatus           
			,	ST4.CorePortfolio
			,	ST4.AssignmentChannel
			,	ST4.AssignmentChannelDetailed
			,	ST4.AssignmentSegmentDetailed
			,	ST4.COA
			,	ST4.Anchor
			,	ST4.SavedDay
			,	ST4.`DOUBLE TREATMENT DNC-HUBS` AS DoubleTreatmentDNCHubs
			,	ST4.OutflowSaved
			,	ST4.RemainingDebt
			,	ST4.FinalRiskSaved
			,	ST4.FinalSaved
			,	ST4.SavedReason
			,	ST4.BU
			,	ST4.CIF
			,	ST4.CURRENT_BUCKET         AS CurrentBucket
			,	ST4.OBJECTIVE_STATUS       AS ObjectiveStatus
			,	ST4.OBJECTIVE_STATUS_DATE  AS ObjectiveStatusDate
			,	ST4.SOM_DEBT             AS SoMDebt
			,	ST4.SOM_BALANCE          AS SoMBalance 
			,	ST4.SOM_BUCKET             AS SomBucket
			,	ST4.InflowType
			,	ST4.Installment
			,	ST4.Outflow
			,	ST4.RiskBucket
			,	ST4.cat
			,	ST4.Target_Type   
			,	ST4.Target_Type_2   
			,	ST4.Payments
			,	ST4.SavedBalance
			,	ST4.`EBA&CAT`
			,	ST4.InternalTeams  
			,	ST4.InternalTeamCode
			,	ST4.`Target Outflow (%)`
			,	ST4.Target_Outflow_Balance
			,	ST4.`DT Target Collection`
			,	ST4.New_Target_Collection_Balance
			,	ST4.External_Target_Rate 
			,	ST4.External_Target
			,	ST4.New_Segment
			,	ST4.Final_Segment
			,	ST4.Assign
			,	ST4.AssignDetail
			,	ST4.`Target %`
			,	ST4.`Target Balance`
			,	ST4.`GIA PINAKA 2`
			,	CASE
					WHEN UF.AccountNumber IS NOT NULL THEN 'Positive to Modify' 
					-- if account has active application then is positive to modify (even if activity is negative)
					WHEN SA.StrongestAction IS NULL AND DA.ExposureKey IS NOT NULL THEN 'Attempt' --if no activty but has dialer then attempt
					WHEN SA.StrongestAction IS NULL THEN 'Not Found' -- if no activity then not found (due serial execution of case-when, account has no active application and no dialer as well)
					WHEN SA.StrongestAction = 'Positive Communication' AND 2ndLevelCategory = '5.1 Positive to Modify' THEN 'Positive to Modify'
					WHEN SA.StrongestAction = 'No Attempt' AND SA.ReportingCategory = '1.1 no contact - no attempt' THEN 'Not Found'
					WHEN SA.StrongestAction = 'No Attempt' AND SA.ReportingCategory = '1.2 no contact - attempt' THEN 'Not Found'  --check with Ari
					ELSE SA.StrongestAction
				END AS Description
			,  SA.PromiseStatus            AS PromiseStatus
		  ,  EX.External_Target_Rate_Calc 
	    ,  UF.AccountNumber           AS UnifiedAccNum
		  ,  SA.StrongestAction         AS StrongestActivity
		  ,  DA.ExposureKey          AS DialerAccNum
		  ,  SA.2ndLevelCategory 
		  ,  SA.ReportingCategory
			,  SA.InteractionDate ActivityDate  
			,  ST4.External_Target_Type_Calc 
			,  ST4.FinalAssignmentChannel
			,  ST4.FinalAssignmentChannelDetailed
		FROM DataSet_step4 ST4
		LEFT JOIN StrongestActivity SA ON ST4.ExposureKey = SA.ExposureKey
		LEFT JOIN Report_Unified	 UF	ON UF.AccountNumber = ST4.ExposureKey
		LEFT JOIN dialer	 DA ON ST4.ExposureKey = DA.ExposureKey
		LEFT JOIN ExternalTargetRateCalc EX ON ST4.External_Target_Type_Calc = EX.External_Target_Type_Calc AND ST4.FinalAssignmentChannel = 'Outsourcing Collection' -- NEW ADD VB 30.10.23
'''
)
DataSet_step5.createOrReplaceTempView("DataSet_step5")

In [0]:
DataSet_step6 = spark.sql(
f'''
        SELECT DISTINCT 
		        ST5.ExposureKey
			,   ST5.EBAStatus
			,   ST5.CorePortfolio
			,   ST5.AssignmentChannel
			,   ST5.AssignmentChannelDetailed
			,   ST5.AssignmentSegmentDetailed
			,   ST5.COA
			,   ST5.Anchor
			,   ST5.SavedDay
			,   ST5.DoubleTreatmentDNCHubs
			,   ST5.OutflowSaved
			,   ST5.RemainingDebt
			,   ST5.FinalRiskSaved
			,   ST5.FinalSaved
			,   ST5.SavedReason
			,   ST5.BU
			,   ST5.CIF
			,   ST5.CurrentBucket
			,   ST5.ObjectiveStatus
			,   ST5.ObjectiveStatusDate
			,   ST5.SoMDebt
			-- ,   ST5.SoMBalance
			,   CASE WHEN CAST(REPLACE(CASE WHEN ST5.SoMBalance IS NULL THEN NULL ELSE ST5.SoMBalance END, ',', '0.') AS NUMERIC(19,6)) > 1 
					 THEN CAST(REPLACE(CASE WHEN ST5.SoMBalance IS NULL THEN NULL ELSE ST5.SoMBalance END, ',', '.') AS NUMERIC(19,6)) 
				ELSE CAST(REPLACE(CASE WHEN ST5.SoMBalance IS NULL THEN NULL ELSE ST5.SoMBalance END, ',', '0.') AS NUMERIC(19,6)) END AS SoMBalance
			,   ST5.SomBucket
			,   ST5.InflowType
			-- ,   ST5.Installment
			,   CASE WHEN CAST(REPLACE(ST5.Installment, ',', '0.') AS NUMERIC(19,6)) > 1 
					 THEN CAST(REPLACE(ST5.Installment, ',', '.') AS NUMERIC(19,6)) 
				ELSE CAST(REPLACE(ST5.Installment, ',', '0.') AS NUMERIC(19,6)) END AS Installment
			,   ST5.Outflow
			,   ST5.RiskBucket
			,   ST5.cat
			,   ST5.Target_Type   
			,   ST5.Target_Type_2   
			,   ST5.Payments
			,   ST5.SavedBalance
			,   ST5.`EBA&CAT`
			,   ST5.InternalTeams  
			,   ST5.InternalTeamCode
			,   ST5.`Target Outflow (%)`
			,   ST5.Target_Outflow_Balance
			,   ST5.`DT Target Collection`
			,   ST5.New_Target_Collection_Balance
			,   ST5.External_Target_Rate 
			,   ST5.External_Target
			,   ST5.New_Segment
			,   ST5.Final_Segment
			,   ST5.Assign
			,   ST5.AssignDetail
			,   ST5.`Target %`
			,   ST5.`Target Balance`
			,   ST5.`GIA PINAKA 2`
			,   ST5.Description
          --   ,   CASE WHEN ST5.Description <> 'Positive Communication' THEN NULL 
       	  --       WHEN ST5.Description = 'Positive Communication' AND ST5.PromiseStatus IS NOT NULL THEN ST5.PromiseStatus
       		--     ELSE CASE WHEN P.InteractionDate = ST5.ActivityDate AND P.PromiseToPayAmount >= P.PromiseAmount  THEN 'Kept'
       		--            WHEN P.InteractionDate = ST5.ActivityDate AND P.PromiseToPayAmount < P.PromiseAmount  THEN 'Broken'
       		-- 		   WHEN P.InteractionDate = ST5.ActivityDate AND P.PromiseDueDate > {run_date!r} THEN 'Pending'
       		-- 		   ELSE ST5.PromiseStatus END 
       		--  END AS PromiseStatus
			, ST5.PromiseStatus
			, ST5.ActivityDate, ST5.External_Target_Type_Calc, ST5.External_Target_Rate_Calc, ST5.UnifiedAccNum, ST5.StrongestActivity, ST5.DialerAccNum, ST5.2ndLevelCategory , ST5.ReportingCategory
			, ST5.FinalAssignmentChannel
			, ST5.FinalAssignmentChannelDetailed
--    , p.InteractionDate, p.PromiseToPayAmount, P.PromiseAmount, P.PromiseDueDate, p.InteractionTypeKey, p.InteractionType
        FROM DataSet_step5 ST5
        -- LEFT JOIN interaction P ON P.ExposureKey = ST5.ExposureKey AND YEAR(P.InteractionDate)=YEAR({run_date!r}) AND MONTH(P.InteractionDate)=MONTH({run_date!r})
'''
)
DataSet_step6.createOrReplaceTempView("DataSet_step6")

In [0]:
DataSet_step7 = spark.sql(
f'''
        SELECT  DISTINCT
                ST6.ExposureKey
			,	ST6.EBAStatus
			,	ST6.CorePortfolio
			,	ST6.AssignmentChannel
			,	ST6.AssignmentChannelDetailed
			,	ST6.AssignmentSegmentDetailed
			,	ST6.COA
			,	ST6.Anchor
			,	ST6.SavedDay
			,	ST6.DoubleTreatmentDNCHubs
			,	ST6.OutflowSaved
			,	ST6.RemainingDebt
			,	ST6.FinalRiskSaved
			,	ST6.FinalSaved
			,	ST6.SavedReason
			,	ST6.BU
			,	ST6.CIF
			,	ST6.CurrentBucket
			,	ST6.ObjectiveStatus
			,	ST6.ObjectiveStatusDate
			,	ST6.SoMDebt
			,	ST6.SoMBalance 
			,	ST6.SomBucket
			,	ST6.InflowType
			,	ST6.Installment
			,	ST6.Outflow
			,	ST6.RiskBucket
			,	ST6.cat
			,	ST6.Target_Type   
			,	ST6.Target_Type_2   
			,	ST6.Payments
			,	ST6.SavedBalance
			,	ST6.`EBA&CAT`
			,	ST6.InternalTeams  
			,	ST6.InternalTeamCode
			,	ST6.`Target Outflow (%)` 
			,	ST6.Target_Outflow_Balance
			,	ST6.`DT Target Collection`
			,	ST6.New_Target_Collection_Balance
			,	ST6.External_Target_Rate 
			,	ST6.External_Target
			,	ST6.New_Segment
			,	ST6.Final_Segment
			,	ST6.Assign
			,	ST6.AssignDetail
			,	ST6.`Target %`
			,	ST6.`Target Balance`
			,	ST6.`GIA PINAKA 2`
			,	ST6.Description
			,	ST6.PromiseStatus AS Description2
			,   CASE WHEN ST6.UnifiedAccNum IS NOT NULL     THEN ST6.SoMBalance
			         WHEN ST6.DialerAccNum IS NOT NULL		THEN ST6.SoMBalance
			         WHEN ST6.StrongestActivity IS NULL		THEN 0
			         WHEN ST6.StrongestActivity = 'No Attempt' AND ST6.ReportingCategory = '1.1 no contact - no attempt' THEN 0
			         ELSE ST6.SoMBalance
			    END AS AttemptedBalance
			,   CASE WHEN ST6.UnifiedAccNum IS NOT NULL		THEN 0
				 	 WHEN ST6.DialerAccNum IS NOT NULL		THEN 0
					 WHEN ST6.StrongestActivity IS NULL		THEN ST6.SoMBalance
					 WHEN ST6.StrongestActivity = 'No Attempt' AND ST6.ReportingCategory = '1.2 no contact - attempt' THEN ST6.SoMBalance
					 ELSE 0
			    END AS NotFoundBalance
			,   CASE WHEN ST6.UnifiedAccNum IS NOT NULL	    THEN ST6.SoMBalance
					 WHEN ST6.DialerAccNum IS NOT NULL      THEN ST6.SoMBalance
					 WHEN ST6.StrongestActivity IS NULL	    THEN 0
					 WHEN ST6.StrongestActivity NOT IN ('No Attempt', 'Attempt') THEN ST6.SoMBalance
		  			 ELSE 0
		    	END AS ContactedBalance
			,   CASE WHEN ST6.UnifiedAccNum IS NOT NULL		THEN ST6.SoMBalance
					 WHEN ST6.StrongestActivity IS NULL       THEN 0
					 WHEN ST6.StrongestActivity IN ('Positive Communication', 'Negative Communication', 'Rest RPC Communication') THEN ST6.SoMBalance
					 ELSE 0
			    END AS RPCBalance
			,   CASE WHEN ST6.StrongestActivity IS NULL	    THEN 0
					 WHEN ST6.StrongestActivity = 'Positive Communication' AND `2ndLevelCategory` = '5.2 Promise to Pay'  THEN ST6.SoMBalance
					 ELSE 0
			    END AS PromiseBalance
			,   CASE WHEN ST6.UnifiedAccNum IS NOT NULL	    THEN ST6.SoMBalance
					 WHEN ST6.StrongestActivity IS NULL		THEN 0
					 WHEN ST6.StrongestActivity = 'Positive Communication' AND `2ndLevelCategory` = '5.1 Positive to Modify' THEN ST6.SoMBalance
					 ELSE 0
			    END AS PositivetoModifyBalance
			,   CASE WHEN ST6.StrongestActivity IS NULL	    THEN 0
					 WHEN ST6.StrongestActivity = 'Negative Communication' AND ST6.UnifiedAccNum IS  NULL THEN ST6.SoMBalance --- VB 9.11.23 ADDED  
					 ELSE 0
			    END AS NegativeBalance
			,   CASE WHEN ST6.StrongestActivity IS NULL       THEN 0
					 WHEN ST6.StrongestActivity = 'Rest RPC Communication' AND ST6.UnifiedAccNum IS  NULL  THEN ST6.SoMBalance--- VB 9.11.23 ADDED  
					 ELSE 0
			    END AS RestRPCCommunicationsBalance
			,   CASE WHEN ST6.PromiseStatus = 'Kept'	    THEN ST6.SoMBalance END	AS `Kept Balance`
			,   CASE WHEN ST6.PromiseStatus = 'Broken'	THEN ST6.SoMBalance END	AS `Broken Balance`
			,   CASE WHEN ST6.PromiseStatus = 'Pending'	THEN ST6.SoMBalance END	AS `Pending Balance`
			,   CASE WHEN ST6.PromiseStatus = 'Cancelled'	THEN ST6.SoMBalance END	AS `Cancelled Balance`
			,   ST6.External_Target_Rate_Calc  
			,   ST6.ActivityDate
			,	ST6.FinalAssignmentChannel
			,	ST6.FinalAssignmentChannelDetailed
		 FROM DataSet_step6 ST6
'''
)
DataSet_step7.createOrReplaceTempView("DataSet_step7")

In [0]:
DataSetFinal = spark.sql(
f'''
    SELECT  DISTINCT
           {run_date!r} AS ReferenceDate 
   		,   C.ExposureKey
   		,   C.EBAStatus
   		,   C.CorePortfolio
        ,   C.AssignmentChannel
        ,   C.AssignmentChannelDetailed
        ,   C.AssignmentSegmentDetailed
        ,   C.COA
        ,   C.Anchor
        ,   C.SavedDay
        ,   C.DoubleTreatmentDNCHubs
        ,   C.OutflowSaved
        ,   C.RemainingDebt
        ,   C.FinalRiskSaved
        ,   C.FinalSaved
        ,   C.SavedReason
        ,   C.BU
        ,   C.CIF
        ,   C.CurrentBucket
        ,   C.ObjectiveStatus
        ,   C.ObjectiveStatusDate 
        ,   C.SoMDebt
        ,   C.SoMBalance
        ,   C.SomBucket
        ,   C.InflowType
        ,   C.Installment
        ,   C.Outflow
        ,   C.RiskBucket 
        ,   C.cat
        --,   C.Target_Type
        ,   C.Target_Type_2
        ,   C.Payments
        ,   C.SavedBalance
        ,   C.`EBA&CAT`
        ,   C.InternalTeams
        ,   C.InternalTeamCode
        ,   C.`Target Outflow (%)` AS `Target_Outflow_Percentage`
        ,   C.Target_Outflow_Balance
        ,   C.`DT Target Collection` AS `DT_Target_Collection`
        ,   C.New_Target_Collection_Balance
        ,   C.External_Target_Rate
        ,   C.External_Target
        ,   C.New_Segment
        ,   C.Final_Segment
        ,   C.Assign
        ,   C.AssignDetail
        ,   C.`Target %` AS `Target_Percentage`
        ,   C.`Target Balance` AS `Target_Balance`
        ,   C.`GIA PINAKA 2` AS `GIA_PINAKA_2`
        ,   C.Description
        ,   C.Description2
        ,   C.AttemptedBalance
        ,   C.NotFoundBalance
        ,   C.ContactedBalance
        ,   C.RPCBalance
        ,   C.PromiseBalance
        ,   C.PositivetoModifyBalance
        ,   C.NegativeBalance
        ,   C.RestRPCCommunicationsBalance
        ,   C.`Kept Balance` AS `Kept_Balance`
        ,   C.`Broken Balance` AS `Broken_Balance`
        ,   C.`Pending Balance` AS `Pending_Balance`
        ,   C.`Cancelled Balance` AS `Cancelled_Balance`
        ,   C.External_Target_Rate_Calc
        ,   C.ActivityDate
        ,   C.FinalAssignmentChannel
        ,   C.FinalAssignmentChannelDetailed
        ,   a1.AttributionChannel					AS PaymentAtttributionChannel_1
        ,   a1.AttributionTeam						AS PaymentAtttributionChannelDetailed_1
        ,   a1.PaymentAmount							AS PaymentAtttributionAmount_1
        ,   a2.AttributionChannel					AS PaymentAtttributionChannel_2
        ,   a2.AttributionTeam						AS PaymentAtttributionChannelDetailed_2
        ,   a2.PaymentAmount							AS PaymentAtttributionAmount_2
        ,   a3.AttributionChannel					AS PaymentAtttributionChannel_3
        ,   a3.AttributionTeam						AS PaymentAtttributionChannelDetailed_3
        ,   a3.PaymentAmount							AS PaymentAtttributionAmount_3
    FROM DataSet_step7 C
    LEFT JOIN AttributionChannel_Pay a1 ON c.ExposureKey = a1.ExposureKey AND a1.Rank = 1
    LEFT JOIN AttributionChannel_Pay a2 ON c.ExposureKey = a2.ExposureKey AND a2.Rank = 2
    LEFT JOIN AttributionChannel_Pay a3 ON c.ExposureKey = a3.ExposureKey AND a3.Rank = 3
'''
)
DataSetFinal.createOrReplaceTempView("DataSetFinal")

##### final_df

In [0]:
final_df = spark.sql(
f'''
    SELECT  DISTINCT
            CAST({run_date!r}                     AS DATE) AS BusinessDate 
   		,   CAST( ExposureKey                         AS STRING) AS ExposureKey                  
   		,   CAST( EBAStatus                             AS STRING) AS EBAStatus
   		,   CAST( CorePortfolio                         AS STRING) AS CorePortfolio
        ,   CAST( AssignmentChannel                     AS STRING) AS AssignmentChannel
        ,   CAST( AssignmentChannelDetailed             AS STRING) AS AssignmentChannelDetailed
        ,   CAST( AssignmentSegmentDetailed             AS STRING) AS AssignmentSegmentDetailed
        ,   CAST( COA                                   AS STRING) AS COA
        ,   CAST( Anchor                                AS STRING) AS Anchor
        ,   CAST( SavedDay                              AS STRING) AS SavedDay
        ,   CAST( DoubleTreatmentDNCHubs                AS STRING) AS DoubleTreatmentDNCHubs
        ,   CAST( OutflowSaved                          AS STRING) AS OutflowSaved
        ,   CAST( RemainingDebt                         AS DECIMAL (19,2)) AS RemainingDebt
        ,   CAST( FinalRiskSaved                        AS STRING)  AS FinalRiskSaved 
        ,   CAST( FinalSaved                            AS STRING) AS FinalSaved
        ,   CAST( SavedReason                           AS STRING) AS SavedReason
        ,   CAST( BU                                    AS STRING) AS AssetSegment
        ,   CAST( CIF                                   AS STRING) AS CDI 
        ,   CAST( CurrentBucket                         AS INT)    AS CurrentBucket
        ,   CAST( ObjectiveStatus                       AS STRING) AS ObjectiveStatus
        ,   CAST( ObjectiveStatusDate                   AS DATE) AS ObjectiveStatusDate
        ,   CAST( SoMDebt                               AS DECIMAL (19,2)) AS SoMDebt
        ,   CAST( SoMBalance                            AS DECIMAL (19,2)) AS SoMBalance 
        ,   CAST( SomBucket                             AS INT) AS SomBucket
        ,   CAST( InflowType                            AS STRING) AS InflowType
        ,   CAST( Installment                           AS DECIMAL (19,2)) AS Installment
        ,   CAST( Outflow                               AS STRING) AS Outflow
        ,   CAST( RiskBucket                            AS STRING) AS RiskBucket
        ,   CAST( cat                                   AS STRING) AS Cat
        ,   CAST( Target_Type_2                         AS STRING) AS TargetType2
        ,   CAST( Payments                              AS DECIMAL (19,2)) AS Payments
        ,   CAST( SavedBalance                          AS DECIMAL (19,2)) AS SavedBalance
        ,   CAST(`EBA&CAT`                              AS STRING) AS EBACat
        ,   CAST(InternalTeams                          AS STRING) AS InternalTeams
        ,   CAST(InternalTeamCode                       AS STRING) AS InternalTeamCode
        ,   CAST(Target_Outflow_Percentage              AS DECIMAL (19,2)) AS  TargetOutflowPercentage
        ,   CAST(Target_Outflow_Balance                 AS DECIMAL (19,2)) AS  TargetOutflowBalance
        ,   CAST(DT_Target_Collection                   AS DECIMAL (19,2)) AS  DTTargetCollection
        ,   CAST(New_Target_Collection_Balance          AS DECIMAL (19,2)) AS  NewTargetCollectionBalance
        ,   CAST(External_Target_Rate                   AS DECIMAL (19,2)) AS  ExternalTargetRate 
        ,   CAST(External_Target                        AS DECIMAL (19,2)) AS  ExternalTarget 
        ,   CAST(New_Segment                            AS STRING) AS NewSegment
        ,   CAST(Final_Segment                          AS STRING) AS FinalSegment
        ,   CAST(Assign                                 AS STRING) AS Assign
        ,   CAST(AssignDetail                           AS STRING) AS AssignDetail
        ,   CAST(Target_Percentage                      AS DECIMAL (19,2))   AS TargetPercentage
        ,   CAST(Target_Balance                         AS DECIMAL (19,2))   AS  TargetBalance
        ,   CAST(GIA_PINAKA_2                           AS STRING) AS ForTable2
        ,   CAST(Description                            AS STRING) AS Description
        ,   CAST(Description2                           AS STRING) AS Description2
        ,   CAST(AttemptedBalance                       AS DECIMAL (19,2)) AS AttemptedBalance
        ,   CAST(NotFoundBalance                        AS DECIMAL (19,2)) AS NotFoundBalance
        ,   CAST(ContactedBalance                       AS DECIMAL (19,2)) AS ContactedBalance
        ,   CAST(RPCBalance                             AS DECIMAL (19,2)) AS RPCBalance
        ,   CAST(PromiseBalance                         AS DECIMAL (19,2)) AS PromiseBalance
        ,   CAST(PositivetoModifyBalance                AS DECIMAL (19,2)) AS PositivetoModifyBalance
        ,   CAST(NegativeBalance                        AS DECIMAL (19,2)) AS NegativeBalance 
        ,   CAST(RestRPCCommunicationsBalance           AS DECIMAL (19,2)) AS RestRPCCommunicationsBalance
        ,   CAST(Kept_Balance                           AS DECIMAL (19,2)) AS KeptBalance
        ,   CAST(Broken_Balance                         AS DECIMAL (19,2)) AS BrokenBalance
        ,   CAST(Pending_Balance                        AS DECIMAL (19,2)) AS PendingBalance
        ,   CAST(Cancelled_Balance                      AS DECIMAL (19,2)) AS CancelledBalance
        ,   CAST(External_Target_Rate_Calc              AS DECIMAL (19,2)) AS ExternalTargetRateCalc
        ,   CAST(ActivityDate                           AS TIMESTAMP) AS ActivityDate
        ,   CAST(FinalAssignmentChannel                 AS STRING) AS FinalAssignmentChannel
        ,   CAST(FinalAssignmentChannelDetailed         AS STRING) AS FinalAssignmentChannelDetailed
        ,   CAST(PaymentAtttributionChannel_1			AS STRING) AS PaymentAtttributionChannel1
        ,   CAST(PaymentAtttributionChannelDetailed_1	AS STRING) AS PaymentAtttributionChannelDetailed1
        ,   CAST(PaymentAtttributionAmount_1			AS DECIMAL(19,2)) AS  PaymentAtttributionAmount1
        ,   CAST(PaymentAtttributionChannel_2			AS STRING) AS  PaymentAtttributionChannel2
        ,   CAST(PaymentAtttributionChannelDetailed_2	AS STRING) AS  PaymentAtttributionChannelDetailed2
        ,   CAST(PaymentAtttributionAmount_2			AS DECIMAL(19,2)) AS  PaymentAtttributionAmount2
        ,   CAST(PaymentAtttributionChannel_3			AS STRING) AS  PaymentAtttributionChannel3
        ,   CAST(PaymentAtttributionChannelDetailed_3	AS STRING) AS  PaymentAtttributionChannelDetailed_3
        ,   CAST(PaymentAtttributionAmount_3			AS DECIMAL(19,2)) AS  PaymentAtttributionAmount3
        ,   CAST(getdate()                              AS TIMESTAMP) AS InsertTimeStamp
    FROM DataSetFinal 
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_CollectionsReport: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
collections_report = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
collections_report.createOrReplaceTempView("collections_report")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/Collections_Report/"
file_name = "Collections_Report_" +run_date+ ".csv"
collections_report.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)